# 10 — Model Selection and Validation
### The Test Set Is Used Once

---

> **Position in the series:**
> `01_statistical_learning_theory` → `02_pac_and_vc_dimensions` → `03_linear_algebra_and_geometry` → `04_pca` → `05_gaussian_mixture_models` → `06_linear_regression` → `07_logistic_regression` → `08_svm_and_kernel_methods` → `09_decision_trees_random_forest_boosting` → **`10_model_selection_and_validation`**

---

## References

| # | Source |
|---|--------|
| [1] | T. Hastie, R. Tibshirani & J. Friedman — *The Elements of Statistical Learning*, 2nd ed., Springer, 2009 — §7.10 (Cross-Validation), §7.10.2 (The Wrong and Right Way to Do Cross-validation) |
| [2] | S. Shalev-Shwartz & S. Ben-David — *Understanding Machine Learning*, Cambridge, 2014 — Ch. 11 (Model Selection and Validation) |
| [3] | A. Géron — *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow*, O'Reilly — Ch. 1 (Testing and Validating), Ch. 2 (End-to-End Machine Learning Project) |
| [4] | G. C. Cawley & N. L. C. Talbot — *On Over-fitting in Model Selection and Subsequent Selection Bias in Performance Evaluation*, JMLR, 11, 2010 |
| [5] | scikit-learn documentation — *Cross-validation: evaluating estimator performance*, https://scikit-learn.org/stable/modules/cross_validation.html; *Common pitfalls and recommended practices*, https://scikit-learn.org/stable/common_pitfalls.html |

---

## Structure

This closing notebook has no new model and no new theorem. Each section is a **problem** met when training and reporting a model, followed by its **solution**.

0. The Rule in One Page
1. Problem: The Training Score Flatters the Model
2. Problem: Tuning on the Test Set
3. Problem: One Validation Split Is Noisy
4. Problem: The Preprocessing Saw the Validation Data
5. Problem: The Winner's Validation Score Is Optimistic Too
6. The Whole Procedure, From Data to the Reported Number
7. Checklist and Summary

The models come from `07`, `08` and `09`; here they are taken from scikit-learn, because the subject is the procedure around them.

---
## 0. The Rule in One Page

> **The test set is used once:** after every decision has been made, to compute the number that is reported. Nothing is changed because of that number.

A score estimates the performance on new data only if the data it is computed on played **no role in building the model**. Building the model includes fitting the parameters, and also every choice made by looking at a score: hyperparameters, features, preprocessing, the model family, when to stop.

So the data get three roles, and each decision uses only its own part:

| Part | Used for | How often |
|---|---|---|
| **Training** | fitting the parameters | as often as needed |
| **Validation** (carved out of the training part, by K-fold) | choosing between candidates | as often as needed |
| **Test** | the reported number | **once** |

**The procedure.** The rest of the notebook follows this list.

1. Split the data once into training and test. Lock the test part away.
2. Using the training part only, compare the candidates by K-fold cross-validation, with all the preprocessing inside the folds.
3. Choose with a rule fixed in advance.
4. Refit the chosen candidate on the whole training part.
5. Open the test set, compute the score, report it. Stop.

The cell below does step 1 on the breast cancer data used in `09`. To make the rule concrete, the test part goes into a small object, `LockedTestSet`, that hands the data out one time and raises an error at the second request.